<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_14_Targeted_TRACE_Artifact_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, json, re, hashlib, csv, zipfile, io, ast, collections
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

MYDRIVE = Path(os.environ.get('ICHI2027_MYDRIVE', '/content/drive/MyDrive'))
PROJECT = MYDRIVE / 'NeuroFHIR_ICHI2027'
OUT = PROJECT / 'phase14_outputs'
OUT.mkdir(parents=True, exist_ok=True)
SHAREABLE = OUT / 'ICHI2027_Phase14_SHAREABLE.json'
PRIVATE = OUT / 'ICHI2027_Phase14_PRIVATE_TRACE_RECOVERY.json'
assert MYDRIVE.is_dir(), 'Google Drive must be mounted.'

def file_sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def private_path(p):
    return str(Path(p).relative_to(MYDRIVE))

def no_identifiers(name):
    name=Path(str(name).replace('\\','/')).name
    if len(name)>150 or re.search(r'(?i)(?:RID|PTID|subject|participant)[_-]?\d{2,}',name):
        return '<WITHHELD>'
    return name

SKIP={'.git','.ipynb_checkpoints','node_modules','__pycache__','.venv','venv'}


Mounted at /content/drive


In [2]:
prior_options = [
    PROJECT/'phase13_outputs'/'ICHI2027_Phase13_SHAREABLE.json',
    Path('/content/ICHI2027_Phase13_SHAREABLE.json'),
]
env_prior=os.environ.get('ICHI2027_PHASE13_JSON')
if env_prior:
    prior_options.insert(0,Path(env_prior))
prior_file=next((p for p in prior_options if p.is_file()),None)
if prior_file is None:
    from google.colab import files
    uploaded=files.upload()
    names=[n for n in uploaded if n=='ICHI2027_Phase13_SHAREABLE.json']
    if len(names)!=1:
        raise FileNotFoundError('Required original ICHI2027_Phase13_SHAREABLE.json')
    prior_file=Path('/content')/names[0]
    prior_file.write_bytes(uploaded[names[0]])
phase13=json.loads(prior_file.read_text(encoding='utf-8'))
assert phase13.get('phase')=='13_original_TRACE_notebook_artifact_recovery'
assert phase13.get('comparison_results') is None
prior_sha=file_sha256(prior_file)
private_manifest=PROJECT/'phase13_outputs'/'ICHI2027_Phase13_PRIVATE_MANIFEST.json'
prior_private=json.loads(private_manifest.read_text(encoding='utf-8')) if private_manifest.is_file() else {}
if prior_private and prior_private.get('run_utc')!=phase13.get('run_utc'):
    prior_private={}


In [3]:
# Look up the exact real-TRACE notebooks identified by Phase 13, never by unrelated ranking signals.
def priority(name):
    n=name.lower()
    if '20b_neurofhir_trace_real_three_state_gate_calibration_corrected' in n: return 1
    if '20b_neurofhir_trace_corrected_three_state_gate_calibration_self_contained' in n: return 2
    if '23b_neurofhir_trace_final_fhir_integration_reproducibility_fixed' in n: return 3
    if '20b_neurofhir_trace_real_three_state_gate_calibration' in n: return 4
    if '20_neurofhir_trace_gate_calibration_heldout_validation' in n: return 5
    if '19a_neurofhir_trace_evidence_family' in n: return 6
    if '24b_neurofhir_trace_final_landmark' in n: return 7
    return None

public_candidates=[x for x in phase13.get('prior_notebook_candidates',[]) if priority(x.get('filename','')) is not None]
private_index={x['content_sha256']:x for x in prior_private.get('notebook_candidates',[])}
needed_hashes={x['content_sha256'] for x in public_candidates}
locs={}
for digest in needed_hashes:
    rec=private_index.get(digest,{})
    if rec.get('private_relative_path'):
        path=MYDRIVE/rec['private_relative_path']
        if path.is_file(): locs[digest]=path
if len(locs)<len(needed_hashes):
    possible_names={x['filename'] for x in public_candidates}
    for current,dirs,names in os.walk(MYDRIVE):
        dirs[:]=[d for d in dirs if d not in SKIP]
        for name in names:
            if name not in possible_names or not name.lower().endswith('.ipynb'): continue
            p=Path(current)/name
            try: digest=file_sha256(p)
            except (OSError,PermissionError): continue
            if digest in needed_hashes and digest not in locs: locs[digest]=p

target_notebooks=[]
for c in sorted(public_candidates,key=lambda x:(priority(x['filename']),x['filename'],x['content_sha256'])):
    digest=c['content_sha256']
    path=locs.get(digest)
    target_notebooks.append({
        'filename':c['filename'], 'content_sha256':digest,
        'priority':priority(c['filename']),
        'file_present':bool(path),
        'private_relative_path':private_path(path) if path else None,
    })
print({'target_notebooks_confirmed':sum(x['file_present'] for x in target_notebooks),
       'target_notebooks_identified':len(target_notebooks)})


{'target_notebooks_confirmed': 16, 'target_notebooks_identified': 16}


In [4]:
# Static audit only: do not execute old notebooks, whose cells may read or publish private data.
art_names={
    'TRACE_ADNI1_20B_REAL_CORRECTED_GATED_PRIVATE.csv',
    'TRACE_ADNI1_20B_REAL_GATED_PRIVATE.csv',
    'frozen_trace_three_state_gate.json',
    'frozen_trace_gate_config.json',
    'heldout_three_state_trace_predictions.csv',
    'trace20B_input_evaluation_table.parquet',
    'trace20_frozen_evaluation_table.parquet',
    'trace20_complete_evaluation_table.parquet',
    'trace_reproducibility_manifest.json',
    'trace_manifest.json',
    'trace_reliability.json',
    'table_feature_trace_states_q90.csv',
    'table_feature_trace_states_q95_sensitivity.csv',
    'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv',
    'TRACE_ADNI1_repeat_pairs_REAL.csv',
    'UCSFFSL51.csv',
    'UCSFFSL51_*.csv',
}
cell_code_audit=[]
for record in target_notebooks:
    if not record['file_present']: continue
    nbpath=MYDRIVE/record['private_relative_path']
    try: nb=json.loads(nbpath.read_text(encoding='utf-8'))
    except (ValueError,UnicodeError,OSError): continue
    found_literals=set()
    selected_cells=[]
    for cell_index,cell in enumerate(nb.get('cells',[])):
        if cell.get('cell_type')!='code': continue
        src=cell.get('source','')
        src=''.join(src) if isinstance(src,list) else str(src)
        if not src: continue
        try:
            tree=ast.parse(src)
            for node in ast.walk(tree):
                if isinstance(node,ast.Constant) and isinstance(node.value,str):
                    val=Path(node.value.replace('\\','/')).name
                    if (val.lower().endswith(('.csv','.parquet','.json')) and len(val)<160
                        and re.search(r'(?i)(trace|gate|repeat|calibrat|threshold|gated|manifest|ucsffsl)',val)):
                        found_literals.add(val)
        except (SyntaxError,ValueError):
            pass
        has_gate=bool(re.search(r'(?i)(frozen_trace|three_state|review_required|trace_state|reliability_state)',src))
        has_write=bool(re.search(r'(?i)(to_csv\s*\(|to_parquet\s*\(|json\.dump|write_text|zipfile)',src))
        if has_gate and has_write:
            selected_cells.append({
                'cell_index_zero_based':cell_index,
                'source_sha256':hashlib.sha256(src.encode()).hexdigest(),
                'source_code_PRIVATE':src[:15000],
                'artifacts_mentioned':[a for a in sorted(found_literals) if a in src],
            })
    art_names.update(found_literals)
    cell_code_audit.append({
        'filename':record['filename'],
        'content_sha256':record['content_sha256'],
        'private_relative_path':record['private_relative_path'],
        'literal_artifact_names':sorted(found_literals),
        'private_selected_code_cells':selected_cells[:25],
    })
print({'notebooks_code_inspected':len(cell_code_audit),
       'possible_artifact_name_count':len(art_names),
       'private_code_cells_selected':sum(len(x['private_selected_code_cells']) for x in cell_code_audit)})


{'notebooks_code_inspected': 16, 'possible_artifact_name_count': 48, 'private_code_cells_selected': 30}


In [5]:
# Targeted exact-filename search including source archives, rather than re-ranking unrelated projects.
SENTINEL={'CONCEPT.csv','VOCABULARY.csv'}
file_index=collections.defaultdict(list)
zip_paths=[]
raw_csv_candidates=[]
for current,dirs,names in os.walk(MYDRIVE):
    dirs[:]=[d for d in dirs if d not in SKIP]
    for name in names:
        p=Path(current)/name
        low=name.lower()
        if low.endswith('.zip'): zip_paths.append(p)
        if name in art_names:
            file_index[low].append(p)
        if low.endswith(('.csv','.tsv','.parquet')) and ('ucsffsl' in low or '20b_real' in low or 'trace' in low and 'gated' in low):
            raw_csv_candidates.append(p)

zip_matches=[]
for archive in zip_paths:
    try:
        with zipfile.ZipFile(archive) as z:
            matches=[]
            for entry in z.infolist():
                if entry.is_dir() or entry.file_size>500*1024*1024: continue
                b=Path(entry.filename).name
                if b.lower() in {x.lower() for x in art_names} or b.upper() in SENTINEL:
                    matches.append({'basename':no_identifiers(b), 'size':entry.file_size,
                                    'member_name_PRIVATE':entry.filename})
            if matches:
                zip_matches.append({'archive_name':no_identifiers(archive.name),
                                    'archive_path_PRIVATE':private_path(archive),
                                    'matching_members':matches})
    except (OSError,zipfile.BadZipFile,RuntimeError):
        continue
print({'matching_standalone_artifacts':sum(map(len,file_index.values())),
       'matching_archives':len(zip_matches),
       'possible_raw_export_names':len(raw_csv_candidates)})


{'matching_standalone_artifacts': 0, 'matching_archives': 0, 'possible_raw_export_names': 0}


In [6]:
# Inspect only schemas and documented links. Do not copy source rows or reproduce TRACE states.
def csv_header(path):
    with Path(path).open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
        first=f.readline(500000)
    if not first: return []
    delimiter='\t' if first.count('\t')>first.count(',') else ','
    return [str(x).strip() for x in next(csv.reader([first],delimiter=delimiter))]

def archive_csv_header(archive, member):
    with zipfile.ZipFile(archive) as z:
        info=z.getinfo(member)
        if info.file_size>500*1024*1024: return []
        with z.open(info) as f:
            header=f.readline(500000).decode('utf-8-sig',errors='replace')
    delimiter='\t' if header.count('\t')>header.count(',') else ','
    return [str(x).strip() for x in next(csv.reader([header],delimiter=delimiter))] if header else []

def inspect_schema(cols):
    up=[x.upper() for x in cols]
    state=[x for x in cols if re.search(r'(?i)(?:TRACE|RELIAB|GATE).*?(?:STATE|STATUS|DECISION|CLASS)',x)]
    rule=[x for x in cols if re.search(r'(?i)(?:TRACE|GATE|METHOD|RULE).*?(?:VERSION|REVISION)',x)]
    calib=[x for x in cols if re.search(r'(?i)(?:CALIBRAT|THRESHOLD|REPEAT).*?(?:REF|ID|VERSION|HASH)',x)]
    linkage=[x for x in cols if x.upper() in {'RID','PTID','BL_IMAGEUID','M12_IMAGEUID','IMAGEUID','SOURCE_IMAGEUID','PERSON_ID'}]
    raw=[x for x in cols if x.upper() in {'ST29SV','ST88SV','ST24CV','ST83CV','ST40CV','ST99CV','ST10CV'}]
    return {'column_count':len(cols),'state_columns':state,'rule_columns':rule,
            'calibration_reference_columns':calib,'join_columns':linkage,'raw_source_fields':raw,
            'candidate_complete_sidecar':bool(state and rule and calib and linkage),
            'candidate_original_raw_export':len(raw)>=5 and bool(linkage)}

artifacts=[]
for basename,paths in sorted(file_index.items()):
    for path in paths[:20]:
        p={'basename':no_identifiers(path.name),'private_relative_path':private_path(path),
           'bytes':path.stat().st_size,'sha256_PRIVATE':file_sha256(path)}
        if path.suffix.lower() in {'.csv','.tsv'}:
            try: p['schema']=inspect_schema(csv_header(path))
            except (OSError,ValueError): p['schema']={'read_error':True}
        elif path.suffix.lower()=='.parquet':
            try:
                import pyarrow.parquet as pq
                p['schema']=inspect_schema(pq.ParquetFile(path).schema_arrow.names)
            except (ImportError,OSError,ValueError): p['schema']={'unavailable':True}
        elif path.suffix.lower()=='.json':
            try:
                with path.open(encoding='utf-8') as f: obj=json.load(f)
                keys=list(obj.keys()) if isinstance(obj,dict) else []
                p['config_key_presence']={
                    'method_or_rule_version':any(re.search(r'(?i)(rule|method|gate).*?(version|revision)',k) for k in keys),
                    'calibration_or_threshold_reference':any(re.search(r'(?i)(calibrat|threshold|repeat)',k) for k in keys),
                    'source_file_hash_or_manifest':any(re.search(r'(?i)(source|sha256|manifest|input)',k) for k in keys),
                }
            except (OSError,ValueError,UnicodeError): p['config_key_presence']={'unreadable':True}
        artifacts.append(p)

for entry in zip_matches:
    for item in entry['matching_members']:
        if not item['basename'].lower().endswith(('.csv','.tsv')): continue
        try: item['schema']=inspect_schema(archive_csv_header(MYDRIVE/entry['archive_path_PRIVATE'],item['member_name_PRIVATE']))
        except (OSError,ValueError,KeyError,zipfile.BadZipFile): item['schema']={'unavailable':True}


In [7]:
# Distinguish an actual original sidecar from code references or historical figures.
standalone_sidecars=[x for x in artifacts if x.get('schema',{}).get('candidate_complete_sidecar')]
archive_sidecars=[{'archive':z['archive_name'],'member':m['basename']}
    for z in zip_matches for m in z['matching_members'] if m.get('schema',{}).get('candidate_complete_sidecar')]
raw_export_candidates=[x for x in artifacts if x.get('schema',{}).get('candidate_original_raw_export')]
archive_raw=[{'archive':z['archive_name'],'member':m['basename']}
    for z in zip_matches for m in z['matching_members'] if m.get('schema',{}).get('candidate_original_raw_export')]
known_config=['frozen_trace_three_state_gate.json','frozen_trace_gate_config.json','trace_reproducibility_manifest.json']
config_candidates=[x for x in artifacts if x['basename'].lower() in known_config]
original_trace_files=[x for x in artifacts if x['basename'].lower().startswith('trace_adni1_20b_real')]

# Final verification intentionally remains false until source-file hashes, original saved run,
# frozen calibration and cohort identifiers can all be reconciled in a dedicated gated run.
verification={
    'original_state_rule_calibration_and_join_in_one_candidate':bool(standalone_sidecars or archive_sidecars),
    'has_frozen_config_file_candidate':bool(config_candidates),
    'has_raw_source_export_header_candidate':bool(raw_export_candidates or archive_raw),
    'full_original_TRACE_provenance_verified':False,
    'source_units_approved':False,
    'DICOM_study_and_series_UIDs_verified':False,
    'athena_vocabulary_verified':False,
    'official_MICDM_load_completed':False,
    'four_arm_comparison_completed':False,
}


In [8]:
private={
    'run_utc':datetime.now(timezone.utc).isoformat(),
    'phase13_json_sha256':prior_sha,
    'target_notebooks':target_notebooks,
    'source_code_audits':cell_code_audit,
    'candidate_artifact_files':artifacts,
    'zip_member_matches':zip_matches,
    'raw_export_filename_candidates':[private_path(p) for p in raw_csv_candidates[:100]],
    'verification':verification,
}
PRIVATE.write_text(json.dumps(private,indent=2,ensure_ascii=False),encoding='utf-8')

public={
    'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
    'phase':'14_targeted_original_TRACE_artifact_recovery',
    'run_utc':private['run_utc'],
    'status':'ACTUAL_PRIOR_NOTEBOOK_STATIC_AUDIT_ONLY_NO_NEW_ETL_PERFORMANCE_RESULTS',
    'privacy':'Only aggregate inspection results and public artifact basenames; paths, code, original data and hashes remain private.',
    'phase13_sha256':prior_sha,
    'prior_notebook_inventory':{
        'neurofhir_trace_notebooks_identified':len(target_notebooks),
        'verified_original_notebook_files_inspected':len(cell_code_audit),
        'prior_code_cells_with_gate_and_artifact_writes':sum(len(x['private_selected_code_cells']) for x in cell_code_audit),
        'target_notebook_basenames':sorted(set(x['filename'] for x in target_notebooks if x['file_present'])),
    },
    'targeted_artifact_inventory':{
        'candidate_filename_types':len(art_names),
        'matching_standalone_files':len(artifacts),
        'matching_archives':len(zip_matches),
        'standalone_sidecar_column_set_candidates':len(standalone_sidecars),
        'archived_sidecar_column_set_candidates':len(archive_sidecars),
        'frozen_config_file_candidates':len(config_candidates),
        'original_20b_private_output_filename_matches':len(original_trace_files),
        'raw_original_export_header_candidates':len(raw_export_candidates)+len(archive_raw),
        'matching_target_artifact_basenames':sorted(set(x['basename'] for x in artifacts))[:65],
        'archive_matched_basenames':sorted(set(m['basename'] for z in zip_matches for m in z['matching_members']))[:65],
    },
    'gates':verification,
    'original_trace_state_results':None,
    'four_arm_comparator_scores':None,
    'real_micdm_load_results':None,
}
if verification['original_state_rule_calibration_and_join_in_one_candidate'] and verification['has_frozen_config_file_candidate']:
    public['next_gate']='VALIDATE_ORIGINAL_TRACE_FILE_HASH_AND_FROZEN_CONFIG_REFERENCES_AGAINST_SOURCE_FILES'
elif cell_code_audit:
    public['next_gate']='REVIEW_VERIFIED_20B_NOTEBOOK_PRIVATE_CODE_AND_REGENERATE_FROZEN_TRACE_OUTPUT_ONLY_IF_AUTHENTIC_INPUTS_EXIST'
else:
    public['next_gate']='PROVIDE_ORIGINAL_20B_TRACE_NOTEBOOK_AND_FROZEN_CONFIG_TO_RECOVER_ORIGINAL_METHOD'
public['external_evidence_still_required']=['ORIGINAL_ADNI_UCSFFSL_EXPORT','FULL_ATHENA_VOCABULARY_PACKAGE']
SHAREABLE.write_text(json.dumps(public,indent=2,ensure_ascii=False),encoding='utf-8')
print({'phase14_shareable_saved':str(SHAREABLE),'phase14_private_saved':str(PRIVATE),
       'verified_prior_notebooks':len(cell_code_audit),
       'original_sidecar_candidates':len(standalone_sidecars)+len(archive_sidecars),
       'next_gate':public['next_gate']})


{'phase14_shareable_saved': '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase14_outputs/ICHI2027_Phase14_SHAREABLE.json', 'phase14_private_saved': '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase14_outputs/ICHI2027_Phase14_PRIVATE_TRACE_RECOVERY.json', 'verified_prior_notebooks': 16, 'original_sidecar_candidates': 0, 'next_gate': 'REVIEW_VERIFIED_20B_NOTEBOOK_PRIVATE_CODE_AND_REGENERATE_FROZEN_TRACE_OUTPUT_ONLY_IF_AUTHENTIC_INPUTS_EXIST'}


In [9]:
try:
    from google.colab import files
    files.download(str(SHAREABLE))
except ImportError:
    print(SHAREABLE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>